In [1]:
# Source Reputation Prediction Pipeline (using 'claim' and 'source_reputation')
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import re
import json

In [2]:
# 1. Load Data from JSONL files (using 'originator' and source metadata)
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

def jsonl_to_df(path):
    data = load_jsonl(path)
    # Use 'originator' as the source identifier
    df = pd.DataFrame(data)
    # If 'originator' is missing, fill with 'unknown'
    if 'originator' not in df.columns:
        df['originator'] = 'unknown'
    return df[['originator']]

train_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/train2.jsonl')
val_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/val2.jsonl')
test_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/test2.jsonl')

In [3]:
# 2. Source Reputation Feature Engineering
source_metadata = {
    # Example metadata for demonstration
    'dwayne-bohac': {'accuracy_score': 0.4, 'awards': 0, 'editorial_transparency': 'low'},
    'scott-surovell': {'accuracy_score': 0.7, 'awards': 1, 'editorial_transparency': 'medium'},
    'barack-obama': {'accuracy_score': 0.9, 'awards': 2, 'editorial_transparency': 'high'},
    'blog-posting': {'accuracy_score': 0.2, 'awards': 0, 'editorial_transparency': 'unknown'},
    # ... add more sources as needed ...
}

def get_source_accuracy(originator):
    return source_metadata.get(originator, {}).get('accuracy_score', -1)

def get_source_awards(originator):
    return source_metadata.get(originator, {}).get('awards', -1)

def get_source_transparency(originator):
    transparency_map = {'low': 0, 'medium': 1, 'high': 2, 'unknown': -1, None: -1}
    val = source_metadata.get(originator, {}).get('editorial_transparency', 'unknown')
    return transparency_map.get(val, -1)

for df in [train_df, val_df, test_df]:
    df['source_accuracy'] = df['originator'].apply(get_source_accuracy)
    df['source_awards'] = df['originator'].apply(get_source_awards)
    df['source_transparency'] = df['originator'].apply(get_source_transparency)

In [4]:
# 3. Embedding Functions (assume these are defined elsewhere)
def generate_embeddings_for_dataframe(df, client, column_name):
    # ...implementation from previous notebooks...
    pass
def explode_embedding_column(df, embedding_col='embedding'):
    # ...implementation from previous notebooks...
    pass

In [5]:
# 4. Generate Embeddings (replace with actual implementation)
# client = ... # initialize your OpenAI client here
for df_name, df in zip(['train_df', 'val_df', 'test_df'], [train_df, val_df, test_df]):
    # Uncomment and use your actual embedding functions
    # df = generate_embeddings_for_dataframe(df, client, 'claim')
    # df = explode_embedding_column(df, 'embedding_claim')
    # globals()[df_name] = df
    pass  # Remove this after implementing embeddings

In [7]:
# 3. Encode Source Reputation Labels (for clustering or classification)
# Here, we treat each unique source as a class label for demonstration.
le = LabelEncoder()
train_df['source_label'] = le.fit_transform(train_df['originator'])

# Map unseen originators in val/test to -1
def safe_transform(le, values):
	classes = set(le.classes_)
	return [le.transform([v])[0] if v in classes else -1 for v in values]

val_df['source_label'] = safe_transform(le, val_df['originator'])
test_df['source_label'] = safe_transform(le, test_df['originator'])

In [8]:
# 4. Prepare Features and Labels for Source Reputation
# Use only source reputation features for prediction/classification
X_train = train_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_train = train_df['source_label'].to_numpy()
X_val = val_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_val = val_df['source_label'].to_numpy()
X_test = test_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_test = test_df['source_label'].to_numpy()

In [ ]:
# 5. Train Only Random Forest Classifier for Source Reputation
classifiers = {
    "Random Forest": RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1)
}

results = {}
try:
    for name, clf in classifiers.items():
        # Check if X_train and y_train are not empty
        if X_train.size == 0 or y_train.size == 0 or X_val.size == 0 or y_val.size == 0:
            print(f"Skipping {name}: Feature or label arrays are empty.")
            continue
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        report = classification_report(y_val, y_pred, target_names=le.classes_)
        results[name] = (acc, report)
        print(f"Classifier: {name}")
        print(f"Accuracy: {acc:.4f}")
        print("Classification Report:")
        print(report)
        print("-" * 60)
except Exception as e:
    print(f"Error during classifier training: {e}")

    ## need to fix here, i think accuracy will improve if this is fixed

Error during classifier training: Number of classes, 447, does not match size of target_names, 2916. Try specifying the labels parameter


In [10]:
# 6. Example Output: Show predictions for first few sources (Random Forest only)
for name, clf in classifiers.items():
    y_pred = clf.predict(X_test)
    pred_labels = le.inverse_transform(y_pred)
    print(f"Classifier: {name}")
    print(test_df[['originator', 'source_accuracy', 'source_awards', 'source_transparency']].assign(predicted_source=pred_labels).head())

# The above code covers:
# - Source reputation prediction
# - Accuracy, awards, and transparency as features
# - Model training and evaluation (Random Forest only)

Classifier: Random Forest
                         originator  source_accuracy  source_awards  \
0                        rick-perry             -1.0             -1   
1                 katrina-shankland             -1.0             -1   
2                      donald-trump             -1.0             -1   
3                     rob-cornilles             -1.0             -1   
4  state-democratic-party-wisconsin             -1.0             -1   

   source_transparency predicted_source  
0                   -1     donald-trump  
1                   -1     donald-trump  
2                   -1     donald-trump  
3                   -1     donald-trump  
4                   -1     donald-trump  


In [13]:
# Prepare features and labels (Random Forest only, source reputation features only)
X_train = train_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_train = train_df['source_label'].to_numpy()
X_val = val_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_val = val_df['source_label'].to_numpy()
X_test = test_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_test = test_df['source_label'].to_numpy()

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred_val = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred_val)
    # Get unique labels in y_val that are not -1 (unseen)
    valid_labels = np.unique(y_val[y_val != -1])
    valid_class_names = le.inverse_transform(valid_labels)
    report = classification_report(
        y_val, y_pred_val,
        labels=valid_labels,
        target_names=valid_class_names
    )
    results[name] = (acc, report)
    print(f"Classifier: {name}")
    print(f"Validation Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(report)
    print("-" * 60)

# Output predictions for test set (Random Forest only, source reputation features only)
for name, clf in classifiers.items():
    y_pred_test = clf.predict(X_test)
    pred_labels = le.inverse_transform(y_pred_test)
    print(f"Classifier: {name}")
    print(test_df[['originator', 'source_accuracy', 'source_awards', 'source_transparency']].assign(predicted_source=pred_labels).head())
    print("-" * 60)

Classifier: Random Forest
Validation Accuracy: 0.0857
Classification Report:
                                             precision    recall  f1-score   support

                        60-plus-association       0.00      0.00      0.00         1
                                    afl-cio       0.00      0.00      0.00         1
                                     afscme       0.00      0.00      0.00         1
                               alan-grayson       0.00      0.00      0.00         2
                                  alex-sink       0.00      0.00      0.00         1
                                 allan-fung       0.00      0.00      0.00         1
                                allen-peake       0.00      0.00      0.00         1
                                 allen-west       0.00      0.00      0.00         1
                               allison-tant       0.00      0.00      0.00         1
                               amanda-fritz       0.00      0.00      0.

/Users/danielbirman/miniforge3/envs/assignment1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/miniforge3/envs/assignment1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/miniforge3/envs/assignment1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [15]:
# --- Source Reputation Metadata Analysis ---
# This cell demonstrates how to integrate source metadata for reputation analysis.

# Example: Define source metadata (in practice, load from a file/database)
source_metadata = {
    # originator: {accuracy_score, awards, editorial_transparency}
    'dwayne-bohac': {'accuracy_score': 0.4, 'awards': 0, 'editorial_transparency': 'low'},
    'scott-surovell': {'accuracy_score': 0.7, 'awards': 1, 'editorial_transparency': 'medium'},
    'barack-obama': {'accuracy_score': 0.9, 'awards': 2, 'editorial_transparency': 'high'},
    'blog-posting': {'accuracy_score': 0.2, 'awards': 0, 'editorial_transparency': 'unknown'},
    # ... add more sources as needed ...
}

# Map metadata to each claim in the training/validation/test sets
for df in [train_df, val_df, test_df]:
    if 'originator' in df.columns:
        df['source_accuracy'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('accuracy_score', None))
        df['source_awards'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('awards', None))
        df['source_editorial_transparency'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('editorial_transparency', None))
    else:
        # If 'originator' column is missing, fill with default values
        df['source_accuracy'] = -1
        df['source_awards'] = -1
        df['source_editorial_transparency'] = 'unknown'

# Example: Print summary statistics for source reputation features
print('Source accuracy scores:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_accuracy']].drop_duplicates().head())
else:
    print(train_df[['source_accuracy']].drop_duplicates().head())
print('\nSource awards:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_awards']].drop_duplicates().head())
else:
    print(train_df[['source_awards']].drop_duplicates().head())
print('\nSource editorial transparency:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_editorial_transparency']].drop_duplicates().head())
else:
    print(train_df[['source_editorial_transparency']].drop_duplicates().head())

# Example: Use these features in model training (add to X_train, X_val, X_test)
# For demonstration, fill missing values with -1 or 'unknown' and encode transparency
transparency_map = {'low': 0, 'medium': 1, 'high': 2, 'unknown': -1, None: -1}
for df in [train_df, val_df, test_df]:
    df['source_accuracy'] = df['source_accuracy'].fillna(-1)
    df['source_awards'] = df['source_awards'].fillna(-1)
    df['source_editorial_transparency_num'] = df['source_editorial_transparency'].map(transparency_map)

# Update feature matrices to include new metadata (Random Forest only)
X_train = train_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_train = train_df['source_label'].to_numpy()
X_val = val_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_val = val_df['source_label'].to_numpy()
X_test = test_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
y_test = test_df['source_label'].to_numpy()

print('Updated feature matrix shape:', X_train.shape)

Source accuracy scores:
       originator  source_accuracy
0    dwayne-bohac              0.4
1  scott-surovell              0.7
2    barack-obama              0.9
3    blog-posting              0.2
4   charlie-crist              NaN

Source awards:
       originator  source_awards
0    dwayne-bohac            0.0
1  scott-surovell            1.0
2    barack-obama            2.0
3    blog-posting            0.0
4   charlie-crist            NaN

Source editorial transparency:
       originator source_editorial_transparency
0    dwayne-bohac                           low
1  scott-surovell                        medium
2    barack-obama                          high
3    blog-posting                       unknown
4   charlie-crist                          None
Updated feature matrix shape: (10269, 3)
